# Phase 5: investigation memory

The assistant remembers what an investigation has established, so the next session starts where the last one stopped; the user can see, edit, export and delete all of it. Full write-up: [`docs/MEMORY.md`](../docs/MEMORY.md).

**Key ideas**
- **Memory item:** one saved conclusion: ruled out (with why), hypothesis, decision, open question or note.
- **Memory block:** the text loaded into the assistant's instructions at the start of every session.
- **Guardrails:** evidence must be real reports or recalls; no personal data; every save is confirmed.
- **Real deletion:** the row is removed and the audit trail keeps no text.

This notebook works in a throwaway schema (`notebook_memory`), so it never touches real investigations.

In [1]:
import json, os, sys
from pathlib import Path
import psycopg
from dotenv import load_dotenv
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')
from mdqc import memory as mem
from mdqc.tools import make_runner
con = psycopg.connect(os.environ['DATABASE_URL'])
S = 'notebook_memory'
con.execute(f'DROP SCHEMA IF EXISTS {S} CASCADE'); mem.ensure_schema(con, S)
inv = mem.create_investigation(con, 'Insulet pod adhesive failures', 'LZG', 'INSULET', schema=S); inv

1

## 1. Session 1 saves what was concluded

These are the same calls the assistant makes through its `remember` tool.

In [2]:
run = make_runner(con, inv, mem.start_session(con, inv, schema=S), S)
for args in [{'kind': 'ruled_out', 'text': 'Rising adhesive failures ruled out for 2022-2024 (down 4.4%).'},
             {'kind': 'hypothesis', 'text': 'Dislodged pods may relate to wear time over 48 hours.'}]:
    print(json.loads(run('remember', args)[0])['confirmation'])

Saved to investigation 1 memory as ruled_out (item 1): "Rising adhesive failures ruled out for 2022-2024 (down 4.4%).". You can view, edit or delete it at any time.
Saved to investigation 1 memory as hypothesis (item 2): "Dislodged pods may relate to wear time over 48 hours.". You can view, edit or delete it at any time.


## 2. Guardrails refuse bad saves

In [3]:
for args in [{'kind': 'note', 'text': 'Compare with this report.', 'evidence': ['MDR 99999999']},
             {'kind': 'note', 'text': 'Call the surgeon on 555-123-4567.'}]:
    print(run('remember', args))

('{"error": "refused: evidence \'MDR 99999999\': no such report in the project data"}', True)
('{"error": "refused: memory must not contain personal contact details (email address or phone number)"}', True)


## 3. Session 2 starts with the memory already loaded

In [4]:
print(mem.memory_block(con, inv, S))

INVESTIGATION MEMORY (investigation 1: Insulet pod adhesive failures; product LZG; manufacturer INSULET; open)
Ruled out:
- [item 1, open] Rising adhesive failures ruled out for 2022-2024 (down 4.4%).
Hypotheses:
- [item 2, open] Dislodged pods may relate to wear time over 48 hours.


## 4. The user deletes the hypothesis: it is gone everywhere

In [5]:
hyp = [i for i in mem.get_items(con, inv, S) if i['kind'] == 'hypothesis'][0]
print(mem.delete_item(con, hyp['item_id'], schema=S)['confirmation'])
print(mem.memory_block(con, inv, S))
hits = [(t, c) for t, c in con.execute("SELECT table_name, column_name FROM information_schema.columns WHERE table_schema = %s AND data_type IN ('text', 'jsonb')", (S,)).fetchall()
        if con.execute(f'SELECT count(*) FROM {S}.{t} WHERE "{c}"::text ILIKE %s', ('%wear time%',)).fetchone()[0]]
print('columns still holding the deleted text:', hits)

Deleted memory item 2; it is no longer stored anywhere.
INVESTIGATION MEMORY (investigation 1: Insulet pod adhesive failures; product LZG; manufacturer INSULET; open)
Ruled out:
- [item 1, open] Rising adhesive failures ruled out for 2022-2024 (down 4.4%).
columns still holding the deleted text: []


## 5. Export: everything stored, nothing hidden

In [6]:
print(mem.export_markdown(con, inv, S))
con.execute(f'DROP SCHEMA {S} CASCADE'); con.commit(); con.close()

# Investigation 1: Insulet pod adhesive failures

Product LZG, manufacturer INSULET, status open; 1 sessions, 1 memory items.

## Ruled out

- **Item 1** (open, by assistant, 2026-11-22): Rising adhesive failures ruled out for 2022-2024 (down 4.4%).

## Audit trail (no text is kept here)

- 2026-11-22T18:49:59 user investigation_created
- 2026-11-22T18:49:59 user session_started
- 2026-11-22T18:49:59 assistant item_created item 1 (ruled_out)
- 2026-11-22T18:49:59 assistant item_created item 2 (hypothesis)
- 2026-11-22T18:49:59 user item_deleted item 2 (hypothesis)



## 6. Checks

In [7]:
V = json.loads((ROOT / 'results/phase5/verify_phase5.json').read_text(encoding='utf-8'))
print('Scorecard', V['scorecard_score'])
[(c['id'], c['check'], 'pass' if c['pass'] else 'FAIL') for c in V['checks']]

Scorecard 5/5


[('M1',
  'memory API round trips, constraints and survival of a data reload (tests on PostgreSQL)',
  'pass'),
 ('M2',
  "each session's memory block = independent replay of all saves/edits/deletes; final DB = model",
  'pass'),
 ('M3',
  'deleted and refused texts appear in no column of any memory table; ids and investigations gone; audit holds no text; scanner catches a planted copy',
  'pass'),
 ('M4',
  'invented evidence, personal data, bad text/kind, closed/missing investigation refused; saves confirmed',
  'pass'),
 ('M5',
  'export (JSON, Markdown) = database; recorded sessions replay offline',
  'pass')]

## Why we did these tasks

| Task | Why we did it | What we learned | What it changes next |
|---|---|---|---|
| Separate memory schema | Memory must survive data reloads and stay apart from FDA data | A data reload leaves memory untouched (tested) | Phase 8's memory page reads this schema |
| Explicit memory tools with confirmations | The user must always know what was saved | Every save is echoed; one investigation cannot touch another | Phase 6 checks the real model confirms its saves |
| Guardrails on evidence and personal data | Memory must not hold invented evidence or personal data | Invented report keys, phones and emails are refused before storing | Phase 7 adds wider guardrail tests |
| Real deletion with a text-free audit trail | Deleting must leave no copy behind | A whole-text scan missed a partial copy; the scan now self-tests | Same pattern for any future store |
| Independent replay of every memory action | To prove each session saw exactly the right memory | All 7 scenario sessions matched | Phase 6 adds multi-session questions for the real model |